# RAGAS Evaluation

This notebook evaluates the Agentic RAG pipeline using the [RAGAS framework](https://docs.ragas.io/).
Because we don't have a labeled ground-truth dataset, we will evaluate using:
- **Faithfulness**: Measures if the generated answer is grounded in the retrieved context (checking for hallucinations).
- **Answer Relevance**: Measures how directly the generated answer addresses the user's initial query.

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")


### 1. Generate Pipeline Outputs
We will run 3 diverse test queries representing our different pipeline branches (Direct LLM, Vectorstore, Web Search).

In [ ]:
from main import agentic_rag_app
from datasets import Dataset

def get_pipeline_data(query: str):
    initial_state = {
        "query": query,
        "original_query": query,
        "namespace": "default_namespace",
        "documents": [],
        "final_context_strips": [],
        "needs_web_search": False,
        "answer": "",
        "retries": 0,
        "route": ""
    }
    result = agentic_rag_app.invoke(initial_state)
    
    # lists for context strings
    context = result.get("final_context_strips", [])
    if not context:
        context = ["Default global knowledge context."]
        
    return result.get("answer", ""), context

queries = [
    "Tell me about the cross-modal architecture of RAG-Anything.",
    "Who won the latest Super Bowl?",
    "What is the specific dual-graph construction technique used? Explain step-by-step."
]

data = {"question": [], "answer": [], "contexts": []}

for q in queries:
    print(f"Evaluating query: {q}")
    ans, ctx = get_pipeline_data(q)
    data["question"].append(q)
    data["answer"].append(ans)
    data["contexts"].append(ctx)

# Create huggingface dataset formatting required by RAGAS
dataset = Dataset.from_dict(data)
dataset.to_pandas()

### 2. Configure RAGAS Custom Models
To avoid OpenAI API costs by default, we configure RAGAS to use our Groq LLM and HuggingFace Local Embeddings.

In [ ]:
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from src.generation.generator import llm

# Wrap our models for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# Explicitly assign models to metrics (RAGAS 0.1.X pattern)
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_emb

### 3. Run RAGAS Evaluation

In [ ]:
from ragas import evaluate

print("Running metrics...")
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb,
)

df_results = results.to_pandas()
print("Evaluation complete!")
df_results[["question", "faithfulness", "answer_relevancy"]]